In [2]:
import ee
import geemap

# ========== 初始化 ==========
try:
    ee.Initialize(project='lst-475011')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='lst-475011')

# ========== AOI: Netherlands ==========
country = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filter(ee.Filter.eq("ADM0_NAME", "Netherlands"))
)
AOI = country.geometry()

# ========== 2022 winter: 2021-12-01 to 2022-03-01 exclusive ==========
start = ee.Date.fromYMD(2021, 12, 1)
end = ee.Date.fromYMD(2022, 3, 1)

# ========== 中等严格 QA 掩膜 ==========
def mask_l2_clouds(img):
    qa = img.select("QA_PIXEL")
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    mask = cloud.Or(shadow).Not()
    return img.updateMask(mask)

# ========== ST_B10 -> Celsius ==========
def to_lst_c(img):
    lstK = img.select("ST_B10").multiply(0.00341802).add(149.0)
    lstC = lstK.subtract(273.15).rename("LST_C")

    # 合理范围筛选
    lstC = lstC.updateMask(lstC.gte(-40).And(lstC.lte(40)))

    return lstC.copyProperties(img, img.propertyNames())

# ========== 构建 Landsat 8/9 LST collection ==========
def get_l89_lst_collection(start, end):
    col8 = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(AOI)
        .filterDate(start, end)
        .map(mask_l2_clouds)
        .map(to_lst_c)
    )

    col9 = (
        ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
        .filterBounds(AOI)
        .filterDate(start, end)
        .map(mask_l2_clouds)
        .map(to_lst_c)
    )

    return col8.merge(col9).select("LST_C")

col = get_l89_lst_collection(start, end)

n = col.size().getInfo()
print("Scenes after filtering:", n)

if n == 0:
    raise RuntimeError("No images found for 2022 winter after filters.")

# ========== per-pixel valid observation count ==========
count = col.count().clip(AOI)

# 荷兰冬季云多，建议先用 1；如果覆盖很好，再试 2
MIN_OBS = 1
valid_mask = count.gte(MIN_OBS)

# ========== median composite ==========
lst_median = col.median().clip(AOI).updateMask(valid_mask)

# ========== Export to Google Drive ==========
folder = "GEE_LST"

desc_lst = f"NL_LST_2022_winter_median_L89_moderate_min{MIN_OBS}_30m_RD"
desc_count = f"NL_LST_2022_winter_validCount_L89_moderate_30m_RD"

task1 = ee.batch.Export.image.toDrive(
    image=lst_median,
    description=desc_lst,
    folder=folder,
    fileNamePrefix=desc_lst,
    region=AOI,
    scale=30,
    crs="EPSG:28992",
    maxPixels=1e13
)
task1.start()
print("✅ Export started:", desc_lst)

task2 = ee.batch.Export.image.toDrive(
    image=count,
    description=desc_count,
    folder=folder,
    fileNamePrefix=desc_count,
    region=AOI,
    scale=30,
    crs="EPSG:28992",
    maxPixels=1e13
)
task2.start()
print("✅ Export started:", desc_count)

print("Done. Check GEE Tasks / Google Drive.")

Scenes after filtering: 36
✅ Export started: NL_LST_2022_winter_median_L89_moderate_min1_30m_RD
✅ Export started: NL_LST_2022_winter_validCount_L89_moderate_30m_RD
Done. Check GEE Tasks / Google Drive.
